In [61]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

# Load and prepare data
def load_and_preprocess_data(filepath):
    with open(filepath, 'r', encoding='utf-8') as file:
        text = file.read()

    # Split text by "..."
    sonnets = text.split('.')
    return sonnets

# Load the sonnets from the provided file
sonnets = load_and_preprocess_data('Sonnet.txt')

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sonnets)
total_words = len(tokenizer.word_index) + 1

# Convert text to sequences
input_sequences = []
for line in sonnets:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences and create predictors and label
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

X, y = input_sequences[:,:-1], input_sequences[:,-1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# Define the model
model = Sequential([
    Embedding(total_words, 2000, input_length=max_sequence_len-1),
    Bidirectional(LSTM(150, return_sequences=True)),
    Dropout(0.2),
    Bidirectional(LSTM(100)),
    Dense(total_words, activation='softmax')
])

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Define early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# Train the model with GPU
with tf.device('/GPU:0'):
    history = model.fit(X, y, epochs=100, batch_size=64, validation_split=0.2, callbacks=[early_stopping])

# Save the model
model.save('v4_shakespeare_sonnet_model.h5')

print("Model training complete. Model saved as 'shakespeare_sonnet_model.h5'")


Epoch 1/100
20/20 [==============================] - 11s 256ms/step - loss: 6.3334 - accuracy: 0.0245 - val_loss: 6.3353 - val_accuracy: 0.0000e+00
Epoch 2/100
20/20 [==============================] - 3s 139ms/step - loss: 5.9388 - accuracy: 0.0253 - val_loss: 6.5435 - val_accuracy: 0.0033
Epoch 3/100
20/20 [==============================] - 2s 128ms/step - loss: 5.8249 - accuracy: 0.0465 - val_loss: 6.6978 - val_accuracy: 0.0261
Epoch 4/100
20/20 [==============================] - 2s 88ms/step - loss: 5.7552 - accuracy: 0.0489 - val_loss: 6.7646 - val_accuracy: 0.0261
Epoch 5/100
20/20 [==============================] - 2s 97ms/step - loss: 5.6532 - accuracy: 0.0538 - val_loss: 6.8364 - val_accuracy: 0.0130
Epoch 6/100
20/20 [==============================] - 2s 99ms/step - loss: 5.5418 - accuracy: 0.0571 - val_loss: 6.9493 - val_accuracy: 0.0130
Epoch 7/100
20/20 [==============================] - 2s 97ms/step - loss: 5.4176 - accuracy: 0.0604 - val_loss: 7.0124 - val_accuracy: 0.013

In [56]:
model.summary()

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_6 (Embedding)     (None, 124, 100)          66700     
                                                                 
 lstm_12 (LSTM)              (None, 124, 150)          150600    
                                                                 
 dropout_17 (Dropout)        (None, 124, 150)          0         
                                                                 
 lstm_13 (LSTM)              (None, 100)               100400    
                                                                 
 dense_11 (Dense)            (None, 667)               67367     
                                                                 
Total params: 385067 (1.47 MB)
Trainable params: 385067 (1.47 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [10]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dropout, BatchNormalization, Dense
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [41]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dropout, BatchNormalization, Dense
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split


# Your sonnet dataset as a string
data = """
Unthrifty loveliness, why dost thou spend
Upon thyself thy beauty’s legacy?
Nature’s bequest gives nothing, but doth lend,
And being frank she lends to those are free:
Then, beauteous niggard, why dost thou abuse
The bounteous largess given thee to give?
Profitless usurer, why dost thou use
So great a sum of sums, yet canst not live?
For having traffic with thyself alone,
Thou of thyself thy sweet self dost deceive:
Then how when nature calls thee to be gone,
What acceptable audit canst thou leave?
Thy unused beauty must be tombed with thee,
Which, used, lives th’ executor to be.
...
Those hours, that with gentle work did frame
The lovely gaze where every eye doth dwell,
Will play the tyrants to the very same
And that unfair which fairly doth excel;
For never-resting time leads summer on
To hideous winter, and confounds him there;
Sap checked with frost, and lusty leaves quite gone,
Beauty o’er-snowed and bareness every where:
Then were not summer’s distillation left,
A liquid prisoner pent in walls of glass,
Beauty’s effect with beauty were bereft,
Nor it, nor no remembrance what it was:
But flowers distill’d, though they with winter meet,
Leese but their show; their substance still lives sweet.
...
Then let not winter’s ragged hand deface,
In thee thy summer, ere thou be distill’d:
Make sweet some vial; treasure thou some place
With beauty’s treasure ere it be self-kill’d.
That use is not forbidden usury,
Which happies those that pay the willing loan;
That’s for thyself to breed another thee,
Or ten times happier, be it ten for one;
Ten times thyself were happier than thou art,
If ten of thine ten times refigur’d thee:
Then what could death do if thou shouldst depart,
Leaving thee living in posterity?
Be not self-will’d, for thou art much too fair
To be death’s conquest and make worms thine heir.
...
Lo! in the orient when the gracious light
Lifts up his burning head, each under eye
Doth homage to his new-appearing sight,
Serving with looks his sacred majesty;
And having climb’d the steep-up heavenly hill,
Resembling strong youth in his middle age,
Yet mortal looks adore his beauty still,
Attending on his golden pilgrimage:
But when from highmost pitch, with weary car,
Like feeble age, he reeleth from the day,
The eyes, ’fore duteous, now converted are
From his low tract, and look another way:
So thou, thyself outgoing in thy noon:
Unlook’d, on diest unless thou get a son.
...
Music to hear, why hear’st thou music sadly?
Sweets with sweets war not, joy delights in joy:
Why lov’st thou that which thou receiv’st not gladly,
Or else receiv’st with pleasure thine annoy?
If the true concord of well-tuned sounds,
By unions married, do offend thine ear,
They do but sweetly chide thee, who confounds
In singleness the parts that thou shouldst bear.
Mark how one string, sweet husband to another,
Strikes each in each by mutual ordering;
Resembling sire and child and happy mother,
Who, all in one, one pleasing note do sing:
Whose speechless song being many, seeming one,
Sings this to thee: ‘Thou single wilt prove none.’
...
Is it for fear to wet a widow’s eye,
That thou consum’st thyself in single life?
Ah! if thou issueless shalt hap to die,
The world will wail thee like a makeless wife;
The world will be thy widow and still weep
That thou no form of thee hast left behind,
When every private widow well may keep
By children’s eyes, her husband’s shape in mind:
Look! what an unthrift in the world doth spend
Shifts but his place, for still the world enjoys it;
But beauty’s waste hath in the world an end,
And kept unused the user so destroys it.
No love toward others in that bosom sits
That on himself such murd’rous shame commits.
...
For shame! deny that thou bear’st love to any,
Who for thyself art so unprovident.
Grant, if thou wilt, thou art belov’d of many,
But that thou none lov’st is most evident:
For thou art so possess’d with murderous hate,
That ’gainst thyself thou stick’st not to conspire,
Seeking that beauteous roof to ruinate
Which to repair should be thy chief desire.
O! change thy thought, that I may change my mind:
Shall hate be fairer lodg’d than gentle love?
Be, as thy presence is, gracious and kind,
Or to thyself at least kind-hearted prove:
Make thee another self for love of me,
that beauty still may live in thine or thee.
...
As fast as thou shalt wane, so fast thou grow’st,
In one of thine, from that which thou departest;
And that fresh blood which youngly thou bestow’st,
Thou mayst call thine when thou from youth convertest,
Herein lives wisdom, beauty, and increase;
Without this folly, age, and cold decay:
If all were minded so, the times should cease
And threescore year would make the world away.
Let those whom nature hath not made for store,
Harsh, featureless, and rude, barrenly perish:
Look, whom she best endow’d, she gave thee more;
Which bounteous gift thou shouldst in bounty cherish:
She carv’d thee for her seal, and meant thereby,
Thou shouldst print more, not let that copy die.
...
When I do count the clock that tells the time,
And see the brave day sunk in hideous night;
When I behold the violet past prime,
And sable curls, all silvered o’er with white;
When lofty trees I see barren of leaves,
Which erst from heat did canopy the herd,
And summer’s green all girded up in sheaves,
Borne on the bier with white and bristly beard,
Then of thy beauty do I question make,
That thou among the wastes of time must go,
Since sweets and beauties do themselves forsake
And die as fast as they see others grow;
And nothing ’gainst Time’s scythe can make defence
Save breed, to brave him when he takes thee hence.
...
O! that you were your self; but, love you are
No longer yours, than you yourself here live:
Against this coming end you should prepare,
And your sweet semblance to some other give:
So should that beauty which you hold in lease
Find no determination; then you were
Yourself again, after yourself’s decease,
When your sweet issue your sweet form should bear.
Who lets so fair a house fall to decay,
Which husbandry in honour might uphold,
Against the stormy gusts of winter’s day
And barren rage of death’s eternal cold?
O! none but unthrifts. Dear my love, you know,
You had a father: let your son say so.
...
Not from the stars do I my judgement pluck;
And yet methinks I have astronomy,
But not to tell of good or evil luck,
Of plagues, of dearths, or seasons’ quality;
Nor can I fortune to brief minutes tell,
Pointing to each his thunder, rain and wind,
Or say with princes if it shall go well
By oft predict that I in heaven find:
But from thine eyes my knowledge I derive,
And constant stars in them I read such art
As ‘Truth and beauty shall together thrive,
If from thyself, to store thou wouldst convert’;
Or else of thee this I prognosticate:
‘Thy end is truth’s and beauty’s doom and date.’
...
When I consider everything that grows
Holds in perfection but a little moment,
That this huge stage presenteth nought but shows
Whereon the stars in secret influence comment;
When I perceive that men as plants increase,
Cheered and checked even by the self-same sky,
Vaunt in their youthful sap, at height decrease,
And wear their brave state out of memory;
Then the conceit of this inconstant stay
Sets you most rich in youth before my sight,
Where wasteful Time debateth with Decay
To change your day of youth to sullied night,
And all in war with Time for love of you,
As he takes from you, I engraft you new.
...
But wherefore do not you a mightier way
Make war upon this bloody tyrant, Time?
And fortify yourself in your decay
With means more blessed than my barren rhyme?
Now stand you on the top of happy hours,
And many maiden gardens, yet unset,
With virtuous wish would bear you living flowers,
Much liker than your painted counterfeit:
So should the lines of life that life repair,
Which this, Time’s pencil, or my pupil pen,
Neither in inward worth nor outward fair,
Can make you live yourself in eyes of men.
To give away yourself, keeps yourself still,
And you must live, drawn by your own sweet skill.
...
Who will believe my verse in time to come,
If it were fill’d with your most high deserts?
Though yet heaven knows it is but as a tomb
Which hides your life, and shows not half your parts.
If I could write the beauty of your eyes,
And in fresh numbers number all your graces,
The age to come would say ‘This poet lies;
Such heavenly touches ne’er touch’d earthly faces.’
So should my papers, yellow’d with their age,
Be scorn’d, like old men of less truth than tongue,
And your true rights be term’d a poet’s rage
And stretched metre of an antique song:
But were some child of yours alive that time,
You should live twice,—in it, and in my rhyme.
"""

# Split the data into sonnets
sonnets = data.strip().split("...")

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sonnets)
total_words = len(tokenizer.word_index) + 1

# Create input sequences
input_sequences = []
for sonnet in sonnets:
    token_list = tokenizer.texts_to_sequences([sonnet])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences
max_sequence_len = max([len(seq) for seq in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Split the data into predictors and labels
X, y = input_sequences[:,:-1], input_sequences[:,-1]
y = np.expand_dims(y, axis=-1)
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)




# Define a new model
model = Sequential()
model.add(Embedding(total_words, 300, input_length=max_sequence_len - 1))  # Increased embedding dimension
model.add(Bidirectional(LSTM(512, return_sequences=True)))  # Increased LSTM units
model.add(Dropout(0.4))  # Adjusted dropout rate
model.add(BatchNormalization())
model.add(Bidirectional(LSTM(512)))
model.add(Dropout(0.4))
model.add(BatchNormalization())
model.add(Dense(256, activation='relu'))  # Added a dense layer
model.add(Dropout(0.3))
model.add(Dense(total_words, activation='softmax'))

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])  # Changed optimizer

# Define callbacks
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.0001)
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model
history = model.fit(X_train, y_train, epochs=50, validation_data=(X_val, y_val), batch_size=64, callbacks=[reduce_lr, early_stop])

# Save the model
model.save('less_dataset_sonnet_prediction_model.h5')

# Print model summary
model.summary()


Total unique words in the dataset: 667
Max sequence length: 125
Training data shape: (1237, 124), (1237, 667)
Validation data shape: (310, 124), (310, 667)


In [42]:
print("Sample sonnets:")
for i in range(5):
    print(sonnets[i])

print(f"Total unique words: {total_words}")


Sample sonnets:
Unthrifty loveliness, why dost thou spend
Upon thyself thy beauty’s legacy?
Nature’s bequest gives nothing, but doth lend,
And being frank she lends to those are free:
Then, beauteous niggard, why dost thou abuse
The bounteous largess given thee to give?
Profitless usurer, why dost thou use
So great a sum of sums, yet canst not live?
For having traffic with thyself alone,
Thou of thyself thy sweet self dost deceive:
Then how when nature calls thee to be gone,
What acceptable audit canst thou leave?
Thy unused beauty must be tombed with thee,
Which, used, lives th’ executor to be.


Those hours, that with gentle work did frame
The lovely gaze where every eye doth dwell,
Will play the tyrants to the very same
And that unfair which fairly doth excel;
For never-resting time leads summer on
To hideous winter, and confounds him there;
Sap checked with frost, and lusty leaves quite gone,
Beauty o’er-snowed and bareness every where:
Then were not summer’s distillation left,
A l

In [43]:
print("Sample training labels (y_train):")
for i in range(5):
    print(np.argmax(y_train[i]))


Sample training labels (y_train):
358
179
28
9
36


In [44]:
print("Sample sonnets:")
for i in range(5):
    print(sonnets[i])

print(f"Total unique words: {total_words}")

print("Sample training input (X_train):")
print(X_train[:5])

print("Sample training labels (y_train):")
print(y_train[:5])


Sample sonnets:
Unthrifty loveliness, why dost thou spend
Upon thyself thy beauty’s legacy?
Nature’s bequest gives nothing, but doth lend,
And being frank she lends to those are free:
Then, beauteous niggard, why dost thou abuse
The bounteous largess given thee to give?
Profitless usurer, why dost thou use
So great a sum of sums, yet canst not live?
For having traffic with thyself alone,
Thou of thyself thy sweet self dost deceive:
Then how when nature calls thee to be gone,
What acceptable audit canst thou leave?
Thy unused beauty must be tombed with thee,
Which, used, lives th’ executor to be.


Those hours, that with gentle work did frame
The lovely gaze where every eye doth dwell,
Will play the tyrants to the very same
And that unfair which fairly doth excel;
For never-resting time leads summer on
To hideous winter, and confounds him there;
Sap checked with frost, and lusty leaves quite gone,
Beauty o’er-snowed and bareness every where:
Then were not summer’s distillation left,
A l

In [45]:
max_sequence_len

125

In [46]:
# Define a new model
model = Sequential()
model.add(Embedding(total_words, 300, input_length=max_sequence_len - 1))  # Increased embedding dimension
model.add(Bidirectional(LSTM(512, return_sequences=True)))  # Increased LSTM units
model.add(Dropout(0.4))  # Adjusted dropout rate
model.add(BatchNormalization())
model.add(Bidirectional(LSTM(512)))
model.add(Dropout(0.4))
model.add(BatchNormalization())
model.add(Dense(256, activation='relu'))  # Added a dense layer
model.add(Dropout(0.3))
model.add(Dense(total_words, activation='softmax'))

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])  # Changed optimizer

# Define callbacks
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.0001)
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model
history = model.fit(X_train, y_train, epochs=50, validation_data=(X_val, y_val), batch_size=64, callbacks=[reduce_lr, early_stop])

# Save the model
model.save('less_dataset_sonnet_prediction_model.h5')

# Print model summary
model.summary()

Epoch 1/50
20/20 [==============================] - 15s 358ms/step - loss: 6.6121 - accuracy: 0.0137 - val_loss: 6.2878 - val_accuracy: 0.0258 - lr: 0.0010
Epoch 2/50
20/20 [==============================] - 4s 220ms/step - loss: 6.1124 - accuracy: 0.0202 - val_loss: 6.2958 - val_accuracy: 0.0161 - lr: 0.0010
Epoch 3/50
20/20 [==============================] - 5s 231ms/step - loss: 6.0589 - accuracy: 0.0218 - val_loss: 6.3118 - val_accuracy: 0.0258 - lr: 0.0010
Epoch 4/50
20/20 [==============================] - 4s 208ms/step - loss: 5.9551 - accuracy: 0.0251 - val_loss: 6.3326 - val_accuracy: 0.0258 - lr: 0.0010
Epoch 5/50
20/20 [==============================] - 4s 209ms/step - loss: 5.9116 - accuracy: 0.0275 - val_loss: 6.3383 - val_accuracy: 0.0258 - lr: 2.0000e-04
Epoch 6/50
20/20 [==============================] - 4s 223ms/step - loss: 5.9045 - accuracy: 0.0315 - val_loss: 6.3451 - val_accuracy: 0.0290 - lr: 2.0000e-04
Epoch 7/50
20/20 [==============================] - 4s 207ms/

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 124, 300)          200100    
                                                                 
 bidirectional_4 (Bidirecti  (None, 124, 1024)         3330048   
 onal)                                                           
                                                                 
 dropout_5 (Dropout)         (None, 124, 1024)         0         
                                                                 
 batch_normalization_4 (Bat  (None, 124, 1024)         4096      
 chNormalization)                                                
                                                                 
 bidirectional_5 (Bidirecti  (None, 1024)              6295552   
 onal)                                                           
                                                      

**Test Code**

In [20]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model

In [21]:
# Load the trained model
model = load_model('improved_sonnet_prediction_model.h5')

In [22]:
# Your sonnet dataset as a string
data = """

"""

In [23]:

# Tokenize the text
tokenizer = Tokenizer()



# Fit the tokenizer on the sonnet data
sonnets = data.strip().split("\n\n")
tokenizer.fit_on_texts(sonnets)
total_words = len(tokenizer.word_index) + 1

In [27]:
print("Sample sonnets:")
for i in range(2):
    print(sonnets[i])

print(f"Total unique words: {total_words}")


Sample sonnets:
Unthrifty loveliness, why dost thou spend
Upon thyself thy beauty’s legacy?
Nature’s bequest gives nothing, but doth lend,
And being frank she lends to those are free:
Then, beauteous niggard, why dost thou abuse
The bounteous largess given thee to give?
Profitless usurer, why dost thou use
So great a sum of sums, yet canst not live?
For having traffic with thyself alone,
Thou of thyself thy sweet self dost deceive:
Then how when nature calls thee to be gone,
What acceptable audit canst thou leave?
    Thy unused beauty must be tombed with thee,
    Which, used, lives th’ executor to be.

Total unique words: 1798


In [31]:
max_sequence_len = 130  # Use the value from your dataset preparation step

# Function to predict the next word
def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')  # Pad sequences
    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted, axis=-1)[0]
    return tokenizer.index_word.get(predicted_word_index, '')

In [32]:
# Test the model with sample inputs
sample_inputs = [
    "From fairest creatures we desire"
]

In [33]:
for sample_input in sample_inputs:
    print(f"Input: {sample_input}")
    next_word = predict_next_word(model, tokenizer, sample_input, max_sequence_len)
    print(f"Predicted next word: {next_word}")
    print()


Input: From fairest creatures we desire
Predicted next word: of



In [34]:
# Function to generate text given a seed text
def generate_text(model, tokenizer, seed_text, next_words, max_sequence_len):
    for _ in range(next_words):
        next_word = predict_next_word(model, tokenizer, seed_text, max_sequence_len)
        if next_word == '':
            break
        seed_text += " " + next_word
    return seed_text

# Generate a continuation of the text
seed_text = "From fairest creatures we desire"
generated_text = generate_text(model, tokenizer, seed_text, 10, max_sequence_len)
print(f"Generated text: {generated_text}")


Generated text: From fairest creatures we desire of and and and and and and and and and


**Copilot code**

In [49]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dropout, BatchNormalization, Dense
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Your sonnet data (same as before)
data = """
Unthrifty loveliness, why dost thou spend
Upon thyself thy beauty’s legacy?
Nature’s bequest gives nothing, but doth lend,
And being frank she lends to those are free:
Then, beauteous niggard, why dost thou abuse
The bounteous largess given thee to give?
Profitless usurer, why dost thou use
So great a sum of sums, yet canst not live?
For having traffic with thyself alone,
Thou of thyself thy sweet self dost deceive:
Then how when nature calls thee to be gone,
What acceptable audit canst thou leave?
Thy unused beauty must be tombed with thee,
Which, used, lives th’ executor to be.
...
Those hours, that with gentle work did frame
The lovely gaze where every eye doth dwell,
Will play the tyrants to the very same
And that unfair which fairly doth excel;
For never-resting time leads summer on
To hideous winter, and confounds him there;
Sap checked with frost, and lusty leaves quite gone,
Beauty o’er-snowed and bareness every where:
Then were not summer’s distillation left,
A liquid prisoner pent in walls of glass,
Beauty’s effect with beauty were bereft,
Nor it, nor no remembrance what it was:
But flowers distill’d, though they with winter meet,
Leese but their show; their substance still lives sweet.
...
Then let not winter’s ragged hand deface,
In thee thy summer, ere thou be distill’d:
Make sweet some vial; treasure thou some place
With beauty’s treasure ere it be self-kill’d.
That use is not forbidden usury,
Which happies those that pay the willing loan;
That’s for thyself to breed another thee,
Or ten times happier, be it ten for one;
Ten times thyself were happier than thou art,
If ten of thine ten times refigur’d thee:
Then what could death do if thou shouldst depart,
Leaving thee living in posterity?
Be not self-will’d, for thou art much too fair
To be death’s conquest and make worms thine heir.
...
Lo! in the orient when the gracious light
Lifts up his burning head, each under eye
Doth homage to his new-appearing sight,
Serving with looks his sacred majesty;
And having climb’d the steep-up heavenly hill,
Resembling strong youth in his middle age,
Yet mortal looks adore his beauty still,
Attending on his golden pilgrimage:
But when from highmost pitch, with weary car,
Like feeble age, he reeleth from the day,
The eyes, ’fore duteous, now converted are
From his low tract, and look another way:
So thou, thyself outgoing in thy noon:
Unlook’d, on diest unless thou get a son.
...
Music to hear, why hear’st thou music sadly?
Sweets with sweets war not, joy delights in joy:
Why lov’st thou that which thou receiv’st not gladly,
Or else receiv’st with pleasure thine annoy?
If the true concord of well-tuned sounds,
By unions married, do offend thine ear,
They do but sweetly chide thee, who confounds
In singleness the parts that thou shouldst bear.
Mark how one string, sweet husband to another,
Strikes each in each by mutual ordering;
Resembling sire and child and happy mother,
Who, all in one, one pleasing note do sing:
Whose speechless song being many, seeming one,
Sings this to thee: ‘Thou single wilt prove none.’
...
Is it for fear to wet a widow’s eye,
That thou consum’st thyself in single life?
Ah! if thou issueless shalt hap to die,
The world will wail thee like a makeless wife;
The world will be thy widow and still weep
That thou no form of thee hast left behind,
When every private widow well may keep
By children’s eyes, her husband’s shape in mind:
Look! what an unthrift in the world doth spend
Shifts but his place, for still the world enjoys it;
But beauty’s waste hath in the world an end,
And kept unused the user so destroys it.
No love toward others in that bosom sits
That on himself such murd’rous shame commits.
...
For shame! deny that thou bear’st love to any,
Who for thyself art so unprovident.
Grant, if thou wilt, thou art belov’d of many,
But that thou none lov’st is most evident:
For thou art so possess’d with murderous hate,
That ’gainst thyself thou stick’st not to conspire,
Seeking that beauteous roof to ruinate
Which to repair should be thy chief desire.
O! change thy thought, that I may change my mind:
Shall hate be fairer lodg’d than gentle love?
Be, as thy presence is, gracious and kind,
Or to thyself at least kind-hearted prove:
Make thee another self for love of me,
that beauty still may live in thine or thee.
...
As fast as thou shalt wane, so fast thou grow’st,
In one of thine, from that which thou departest;
And that fresh blood which youngly thou bestow’st,
Thou mayst call thine when thou from youth convertest,
Herein lives wisdom, beauty, and increase;
Without this folly, age, and cold decay:
If all were minded so, the times should cease
And threescore year would make the world away.
Let those whom nature hath not made for store,
Harsh, featureless, and rude, barrenly perish:
Look, whom she best endow’d, she gave thee more;
Which bounteous gift thou shouldst in bounty cherish:
She carv’d thee for her seal, and meant thereby,
Thou shouldst print more, not let that copy die.
...
When I do count the clock that tells the time,
And see the brave day sunk in hideous night;
When I behold the violet past prime,
And sable curls, all silvered o’er with white;
When lofty trees I see barren of leaves,
Which erst from heat did canopy the herd,
And summer’s green all girded up in sheaves,
Borne on the bier with white and bristly beard,
Then of thy beauty do I question make,
That thou among the wastes of time must go,
Since sweets and beauties do themselves forsake
And die as fast as they see others grow;
And nothing ’gainst Time’s scythe can make defence
Save breed, to brave him when he takes thee hence.
...
O! that you were your self; but, love you are
No longer yours, than you yourself here live:
Against this coming end you should prepare,
And your sweet semblance to some other give:
So should that beauty which you hold in lease
Find no determination; then you were
Yourself again, after yourself’s decease,
When your sweet issue your sweet form should bear.
Who lets so fair a house fall to decay,
Which husbandry in honour might uphold,
Against the stormy gusts of winter’s day
And barren rage of death’s eternal cold?
O! none but unthrifts. Dear my love, you know,
You had a father: let your son say so.
...
Not from the stars do I my judgement pluck;
And yet methinks I have astronomy,
But not to tell of good or evil luck,
Of plagues, of dearths, or seasons’ quality;
Nor can I fortune to brief minutes tell,
Pointing to each his thunder, rain and wind,
Or say with princes if it shall go well
By oft predict that I in heaven find:
But from thine eyes my knowledge I derive,
And constant stars in them I read such art
As ‘Truth and beauty shall together thrive,
If from thyself, to store thou wouldst convert’;
Or else of thee this I prognosticate:
‘Thy end is truth’s and beauty’s doom and date.’
...
When I consider everything that grows
Holds in perfection but a little moment,
That this huge stage presenteth nought but shows
Whereon the stars in secret influence comment;
When I perceive that men as plants increase,
Cheered and checked even by the self-same sky,
Vaunt in their youthful sap, at height decrease,
And wear their brave state out of memory;
Then the conceit of this inconstant stay
Sets you most rich in youth before my sight,
Where wasteful Time debateth with Decay
To change your day of youth to sullied night,
And all in war with Time for love of you,
As he takes from you, I engraft you new.
...
But wherefore do not you a mightier way
Make war upon this bloody tyrant, Time?
And fortify yourself in your decay
With means more blessed than my barren rhyme?
Now stand you on the top of happy hours,
And many maiden gardens, yet unset,
With virtuous wish would bear you living flowers,
Much liker than your painted counterfeit:
So should the lines of life that life repair,
Which this, Time’s pencil, or my pupil pen,
Neither in inward worth nor outward fair,
Can make you live yourself in eyes of men.
To give away yourself, keeps yourself still,
And you must live, drawn by your own sweet skill.
...
Who will believe my verse in time to come,
If it were fill’d with your most high deserts?
Though yet heaven knows it is but as a tomb
Which hides your life, and shows not half your parts.
If I could write the beauty of your eyes,
And in fresh numbers number all your graces,
The age to come would say ‘This poet lies;
Such heavenly touches ne’er touch’d earthly faces.’
So should my papers, yellow’d with their age,
Be scorn’d, like old men of less truth than tongue,
And your true rights be term’d a poet’s rage
And stretched metre of an antique song:
But were some child of yours alive that time,
You should live twice,—in it, and in my rhyme.
"""


# Split the data into sonnets
sonnets = data.strip().split("...")

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sonnets)
total_words = len(tokenizer.word_index) + 1

# Create input sequences
input_sequences = []
for sonnet in sonnets:
    token_list = tokenizer.texts_to_sequences([sonnet])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences
max_sequence_len = max([len(seq) for seq in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Split the data into predictors and labels
X, y = input_sequences[:, :-1], input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Define a new model
model = Sequential()
model.add(Embedding(total_words, 300, input_length=max_sequence_len - 1))
model.add(Bidirectional(LSTM(256, return_sequences=True)))
model.add(Dropout(0.4))
model.add(BatchNormalization())
model.add(Bidirectional(LSTM(256)))
model.add(Dropout(0.4))
model.add(BatchNormalization())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(total_words, activation='softmax'))

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Define callbacks
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.0001)
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model
history = model.fit(X_train, y_train, epochs=50, validation_data=(X_val, y_val), batch_size=64, callbacks=[reduce_lr, early_stop])

# Save the model
model.save('sonnet_prediction_model.h5')

# Print model summary
model.summary()



Epoch 1/50
20/20 [==============================] - 12s 248ms/step - loss: 6.5306 - accuracy: 0.0105 - val_loss: 6.4424 - val_accuracy: 0.0323 - lr: 0.0010
Epoch 2/50
20/20 [==============================] - 2s 112ms/step - loss: 6.1108 - accuracy: 0.0243 - val_loss: 6.3140 - val_accuracy: 0.0129 - lr: 0.0010
Epoch 3/50
20/20 [==============================] - 2s 113ms/step - loss: 5.9796 - accuracy: 0.0154 - val_loss: 6.3160 - val_accuracy: 0.0194 - lr: 0.0010
Epoch 4/50
20/20 [==============================] - 2s 95ms/step - loss: 5.9622 - accuracy: 0.0283 - val_loss: 6.3527 - val_accuracy: 0.0258 - lr: 0.0010
Epoch 5/50
20/20 [==============================] - 2s 105ms/step - loss: 5.9113 - accuracy: 0.0251 - val_loss: 6.4463 - val_accuracy: 0.0194 - lr: 0.0010
Epoch 6/50
20/20 [==============================] - 2s 123ms/step - loss: 5.8774 - accuracy: 0.0267 - val_loss: 6.4331 - val_accuracy: 0.0194 - lr: 2.0000e-04
Epoch 7/50
20/20 [==============================] - 2s 104ms/step 